<a href="https://colab.research.google.com/github/madelsu/MOSAIC-Agentic-Severity-Phenotyping/blob/main/study_design/cohort_construction/Final_Four_Index_Dates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# MOSAIC — Patient Excel File Generation
# CELL 1: Setup & Library Imports
# ============================================================
# Generates 4 Excel files (one per index date) for the
# MOSAIC LLM pipeline. Each file contains all 244 eligible
# patients with their clinical data in the 5-year lookback
# window before each index date.
#
# Output files:
#   T2D_patients_FIRST_DX.xlsx
#   T2D_patients_FIRST_TX.xlsx
#   T2D_patients_RANDOM.xlsx
#   T2D_patients_LATEST.xlsx
# ============================================================

!pip install openpyxl --quiet

import os
import warnings
import pandas as pd
import numpy as np
from datetime import datetime

warnings.filterwarnings("ignore")
pd.set_option("future.no_silent_downcasting", True)

def to_naive(series):
    """Parse to datetime and strip timezone if present."""
    s = pd.to_datetime(series, errors="coerce")
    if hasattr(s, "dt") and s.dt.tz is not None:
        s = s.dt.tz_localize(None)
    return s

print("=" * 55)
print("  MOSAIC — Patient Excel File Generation")
print("  Cell 1: Libraries loaded successfully")
print("=" * 55)
print(f"  pandas  {pd.__version__}")
print(f"  numpy   {np.__version__}")
print("=" * 55)

In [ ]:
# ============================================================
# MOSAIC — Patient Excel File Generation
# CELL 2: Paths & Configuration
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# ── Paths ──
BASE_PATH    = "/content/drive/MyDrive/THESIS/Data/Coherent_data/csv/"
COHORT_PATH  = "/content/drive/MyDrive/THESIS/Data/Coherent_data/cohort_outputs/T2D_eligible_cohort.csv"
OUTPUT_PATH  = "/content/drive/MyDrive/THESIS/Data/Coherent_data/patient_excel_files/"

os.makedirs(OUTPUT_PATH, exist_ok=True)

# ── Parameters ──
LOOKBACK_YEARS = 5       # years of data to include before index date
DISEASE_NAME   = "T2D"

# ── Index date column names in the cohort CSV ──
INDEX_DATE_CONFIGS = {
    "FIRST_DX": {
        "index_col":   "INDEX_FIRST_DX",
        "label":       "First Diagnosis",
        "filename":    f"{DISEASE_NAME}_patients_FIRST_DX.xlsx",
        "use_lookback": True,    # filter to 5yr window
    },
    "FIRST_TX": {
        "index_col":   "INDEX_FIRST_TX",
        "label":       "First Treatment",
        "filename":    f"{DISEASE_NAME}_patients_FIRST_TX.xlsx",
        "use_lookback": True,
    },
    "RANDOM": {
        "index_col":   "INDEX_RANDOM",
        "label":       "Random Date",
        "filename":    f"{DISEASE_NAME}_patients_RANDOM.xlsx",
        "use_lookback": True,
    },
    "LATEST": {
        "index_col":   "INDEX_LATEST",
        "label":       "Latest (Cross-sectional)",
        "filename":    f"{DISEASE_NAME}_patients_LATEST.xlsx",
        "use_lookback": False,   # use all available data
    },
}

# ── Clinical CSV files to include as sheets ──
# Maps sheet name → (csv filename, patient column, date columns)
CLINICAL_FILES = {
    "patients":        ("patients.csv",        "Id",      []),
    "conditions":      ("conditions.csv",       "PATIENT", ["START", "STOP"]),
    "observations":    ("observations.csv",     "PATIENT", ["DATE"]),
    "medications":     ("medications.csv",      "PATIENT", ["START", "STOP"]),
    "encounters":      ("encounters.csv",       "PATIENT", ["START", "STOP"]),
    "procedures":      ("procedures.csv",       "PATIENT", ["DATE"]),
    "careplans":       ("careplans.csv",        "PATIENT", ["START", "STOP"]),
    "immunizations":   ("immunizations.csv",    "PATIENT", ["DATE"]),
    "devices":         ("devices.csv",          "PATIENT", ["START", "STOP"]),
    "allergies":       ("allergies.csv",        "PATIENT", ["START", "STOP"]),
    "imaging_studies": ("imaging_studies.csv",  "PATIENT", ["DATE"]),
}

# ── Columns to DROP from patients sheet (PII / financial) ──
PATIENTS_DROP_COLS = [
    "SSN", "DRIVERS", "PASSPORT", "PREFIX", "FIRST", "LAST",
    "SUFFIX", "MAIDEN", "ADDRESS", "CITY", "ZIP", "LAT", "LON",
    "HEALTHCARE_EXPENSES", "HEALTHCARE_COVERAGE",
]

print("=" * 55)
print(f"  MOSAIC — {DISEASE_NAME} Patient Excel Generation")
print("=" * 55)
print(f"  Cohort file : {COHORT_PATH}")
print(f"  Base path   : {BASE_PATH}")
print(f"  Output path : {OUTPUT_PATH}")
print(f"  Lookback    : {LOOKBACK_YEARS} years")
print(f"  Index dates : {list(INDEX_DATE_CONFIGS.keys())}")
print(f"  CSV sheets  : {list(CLINICAL_FILES.keys())}")
print("=" * 55)
print("  Ready — run Cell 3 to load cohort and generate files")
print("=" * 55)

In [ ]:
# ============================================================
# MOSAIC — Patient Excel File Generation
# CELL 3: Load Cohort & Preload Clinical CSVs
# ============================================================
# Loads the eligible cohort and all clinical CSV files into
# memory once — avoids re-reading files 4 times (once per
# index date). All date columns are parsed and made tz-naive.
# ============================================================

# ── Load eligible cohort ──
print("Loading eligible cohort...")

cohort = pd.read_csv(COHORT_PATH, low_memory=False)

# Parse all index date columns
for cfg in INDEX_DATE_CONFIGS.values():
    col = cfg["index_col"]
    if col in cohort.columns:
        cohort[col] = to_naive(cohort[col])

# Standardise patient ID column
if "PATIENT_ID" in cohort.columns:
    cohort = cohort.rename(columns={"PATIENT_ID": "PATIENT"})

patient_ids = set(cohort["PATIENT"].tolist())

print(f"  Cohort loaded: {len(cohort):,} patients")
print(f"  Index date columns found:")
for key, cfg in INDEX_DATE_CONFIGS.items():
    col = cfg["index_col"]
    if col in cohort.columns:
        n_valid = cohort[col].notna().sum()
        print(f"    {key:<12} → {col:<22} ({n_valid:,} non-null)")
    else:
        print(f"    {key:<12} → {col:<22} ← COLUMN NOT FOUND")

# ── Preload all clinical CSV files ──
print(f"\nPreloading {len(CLINICAL_FILES)} clinical CSV files...")
print("-" * 55)

clinical_dfs = {}

for sheet_name, (csv_file, pat_col, date_cols) in CLINICAL_FILES.items():
    filepath = os.path.join(BASE_PATH, csv_file)

    if not os.path.exists(filepath):
        print(f"  MISSING : {csv_file} — skipping")
        clinical_dfs[sheet_name] = pd.DataFrame()
        continue

    df = pd.read_csv(filepath, low_memory=False)

    # Standardise patient ID column for patients.csv
    if pat_col == "Id" and "Id" in df.columns:
        df = df.rename(columns={"Id": "PATIENT"})
        pat_col = "PATIENT"
    elif pat_col == "Id" and "ID" in df.columns:
        df = df.rename(columns={"ID": "PATIENT"})
        pat_col = "PATIENT"

    # Filter to cohort patients only
    if pat_col in df.columns:
        df = df[df[pat_col].isin(patient_ids)].copy()
    else:
        print(f"  WARN    : {csv_file} — patient column '{pat_col}' not found")

    # Parse date columns
    for col in date_cols:
        if col in df.columns:
            df[col] = to_naive(df[col])

    # Drop PII columns from patients sheet
    if sheet_name == "patients":
        cols_to_drop = [c for c in PATIENTS_DROP_COLS if c in df.columns]
        df = df.drop(columns=cols_to_drop)

    clinical_dfs[sheet_name] = df
    print(f"  OK  {csv_file:<30} {len(df):>8,} rows  "
          f"({df['PATIENT'].nunique() if 'PATIENT' in df.columns else '?':,} patients)")

print("-" * 55)
print(f"  All files loaded. Ready to generate Excel files.")
print("=" * 55)

In [ ]:
# ============================================================
# MOSAIC — Patient Excel File Generation
# CELL 4: Generate Excel Files
# ============================================================
# For each index date, produces one Excel file with:
#   Sheet 1 — patient_list   (all patients + index date used)
#   Sheet 2 — patients       (demographics, no PII)
#   Sheet 3 — conditions     (filtered to lookback window)
#   Sheet 4 — observations   (filtered to lookback window)
#   Sheet 5 — medications    (collapsed: 1 row per unique drug)
#   Sheet 6 — encounters     (filtered to lookback window)
#   Sheet 7 — procedures     (filtered to lookback window)
#   Sheet 8 — careplans      (filtered to lookback window)
#   Sheet 9 — immunizations  (filtered to lookback window)
#   Sheet 10 — devices       (filtered to lookback window)
#   Sheet 11 — allergies     (filtered to lookback window)
#   Sheet 12 — imaging_studies (filtered to lookback window)
#
# MEDICATION COLLAPSE:
#   Synthea generates 1 row per annual refill. We collapse to
#   1 row per unique (PATIENT, CODE, DESCRIPTION) with:
#   FIRST_START, LAST_STOP, TOTAL_PRESCRIPTIONS, STATUS, REASONDESCRIPTION
# ============================================================

def filter_to_window(df, date_cols, window_start, window_end):
    """
    Keep rows where ANY date column falls within
    [window_start, window_end] for that patient.
    window_start and window_end are pd.Series indexed by PATIENT.
    """
    if df.empty or not date_cols:
        return df

    # Build a mask — row kept if any date col falls in window
    mask = pd.Series(False, index=df.index)

    for date_col in date_cols:
        if date_col not in df.columns:
            continue
        # Merge window dates onto df via PATIENT
        merged = df[["PATIENT", date_col]].copy()
        merged = merged.merge(
            pd.DataFrame({
                "PATIENT":       window_start.index,
                "_WIN_START":    window_start.values,
                "_WIN_END":      window_end.values,
            }),
            on="PATIENT", how="left"
        )
        col_mask = (
            merged[date_col].notna() &
            (merged[date_col] >= merged["_WIN_START"]) &
            (merged[date_col] <= merged["_WIN_END"])
        )
        mask = mask | col_mask.values

    return df[mask].copy()


def collapse_medications(meds_df):
    """
    Collapse medications from one row per refill to
    one row per unique (PATIENT, CODE, DESCRIPTION).
    Preserves: DESCRIPTION, CODE, FIRST_START, LAST_STOP,
               TOTAL_PRESCRIPTIONS, STATUS, REASONDESCRIPTION

    FIX: Previous version used DESCRIPTION as both a groupby key
    and a count aggregation target, which overwrote DESCRIPTION
    with the count. Now uses a separate _COUNT column.
    """
    if meds_df.empty:
        return meds_df

    meds_df = meds_df.copy()

    # Determine STATUS: active if no STOP date, stopped otherwise
    if "STOP" in meds_df.columns:
        meds_df["_STATUS"] = meds_df["STOP"].isna().map(
            {True: "active", False: "stopped"}
        )
    else:
        meds_df["_STATUS"] = "unknown"

    # Separate count column — avoids consuming DESCRIPTION in agg
    meds_df["_COUNT"] = 1

    # Group by all identifying columns
    group_cols = ["PATIENT"]
    if "CODE" in meds_df.columns:
        group_cols.append("CODE")
    if "DESCRIPTION" in meds_df.columns:
        group_cols.append("DESCRIPTION")

    # Aggregation dict — does NOT touch DESCRIPTION or CODE
    agg_dict = {"_COUNT": "sum"}
    if "START" in meds_df.columns:
        agg_dict["START"] = "min"
    if "STOP" in meds_df.columns:
        agg_dict["STOP"] = "max"
    agg_dict["_STATUS"] = lambda x: (
        "active" if "active" in x.values else "stopped"
    )
    if "REASONDESCRIPTION" in meds_df.columns:
        agg_dict["REASONDESCRIPTION"] = lambda x: (
            x.dropna().mode().iloc[0] if len(x.dropna()) > 0 else ""
        )

    collapsed = meds_df.groupby(group_cols, as_index=False).agg(agg_dict)

    # Rename for clarity
    rename = {"_COUNT": "TOTAL_PRESCRIPTIONS", "_STATUS": "STATUS"}
    if "START" in collapsed.columns:
        rename["START"] = "FIRST_START"
    if "STOP" in collapsed.columns:
        rename["STOP"] = "LAST_STOP"
    collapsed = collapsed.rename(columns=rename)

    return collapsed


def build_patient_list_sheet(cohort, index_col, label):
    """
    Build Sheet 1: patient list with index date and key metadata.
    """
    cols = ["PATIENT", index_col]
    optional = ["GENDER", "RACE", "AGE_AT_DX", "DIED",
                "DEATH_DATE", "OBS_START", "OBS_END"]
    for c in optional:
        if c in cohort.columns:
            cols.append(c)

    sheet = cohort[cols].copy()
    sheet = sheet.rename(columns={index_col: "INDEX_DATE"})
    sheet.insert(2, "INDEX_DATE_TYPE", label)

    # Format date columns as strings
    for col in ["INDEX_DATE", "OBS_START", "OBS_END", "DEATH_DATE"]:
        if col in sheet.columns:
            sheet[col] = pd.to_datetime(
                sheet[col], errors="coerce"
            ).dt.strftime("%Y-%m-%d").fillna("")

    return sheet


# ── MAIN LOOP: generate one Excel per index date ──

lookback_days = LOOKBACK_YEARS * 365.25

for idx_key, cfg in INDEX_DATE_CONFIGS.items():
    index_col   = cfg["index_col"]
    label       = cfg["label"]
    filename    = cfg["filename"]
    use_lookback = cfg["use_lookback"]
    out_path    = os.path.join(OUTPUT_PATH, filename)

    print(f"\n{'='*60}")
    print(f"  Generating: {filename}")
    print(f"  Index date: {label}")
    print(f"  Lookback  : {'5 years before index date' if use_lookback else 'ALL available data'}")
    print(f"{'='*60}")

    if index_col not in cohort.columns:
        print(f"  SKIPPING — column {index_col} not found in cohort")
        continue

    # Build per-patient window boundaries
    index_dates = cohort.set_index("PATIENT")[index_col]

    if use_lookback:
        window_end   = index_dates
        window_start = index_dates - pd.to_timedelta(lookback_days, unit="D")
    else:
        # Latest: use OBS_START → OBS_END (all data)
        window_start = to_naive(cohort.set_index("PATIENT")["OBS_START"])
        window_end   = to_naive(cohort.set_index("PATIENT")["OBS_END"])

    # Write Excel
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:

        # ── Sheet 1: patient_list ──
        print(f"  Writing sheet: patient_list...")
        patient_list = build_patient_list_sheet(cohort, index_col, label)
        patient_list.to_excel(writer, sheet_name="patient_list", index=False)
        print(f"    {len(patient_list):,} rows")

        # ── Remaining sheets: one per clinical CSV ──
        for sheet_name, (csv_file, pat_col, date_cols) in CLINICAL_FILES.items():
            df = clinical_dfs.get(sheet_name, pd.DataFrame())

            if df.empty:
                print(f"  Writing sheet: {sheet_name:<20} EMPTY — skipping")
                continue

            # Apply time window filter
            if date_cols and use_lookback:
                df_filtered = filter_to_window(
                    df, date_cols, window_start, window_end
                )
            elif date_cols and not use_lookback:
                # Latest: filter to OBS_START → OBS_END per patient
                df_filtered = filter_to_window(
                    df, date_cols, window_start, window_end
                )
            else:
                # patients sheet — no date filter, just include all
                df_filtered = df.copy()

            # Collapse medications
            if sheet_name == "medications" and not df_filtered.empty:
                df_filtered = collapse_medications(df_filtered)

            # Format date columns as strings for Excel readability
            for col in df_filtered.select_dtypes(
                include=["datetime64[ns]", "datetimetz"]
            ).columns:
                df_filtered[col] = df_filtered[col].dt.strftime(
                    "%Y-%m-%d"
                ).fillna("")

            # Write sheet
            n_patients = (
                df_filtered["PATIENT"].nunique()
                if "PATIENT" in df_filtered.columns else "?"
            )
            print(f"  Writing sheet: {sheet_name:<20} "
                  f"{len(df_filtered):>7,} rows  "
                  f"({n_patients} patients)")

            df_filtered.to_excel(
                writer, sheet_name=sheet_name, index=False
            )

    print(f"\n  ✓ Saved → {out_path}")

print(f"\n{'='*60}")
print(f"  ALL EXCEL FILES GENERATED")
print(f"{'='*60}")
print(f"  Output folder: {OUTPUT_PATH}")
for cfg in INDEX_DATE_CONFIGS.values():
    fpath = os.path.join(OUTPUT_PATH, cfg['filename'])
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {cfg['filename']:<45} {size_mb:.1f} MB")
print(f"{'='*60}")